CRAG T5 EVALUATOR

Checking GPU...
GPU: Tesla T4
GPU memory: 14.56 GB
CUDA: True

UPLOAD YOUR DATASET


Saving train_popqa.txt to train_popqa (2).txt

Dataset: /content/train_popqa (2).txt

DOWNLOADING T5-LARGE


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

[transformers] T5ForSequenceClassification LOAD REPORT from: t5-large
Key                                 | Status  | 
------------------------------------+---------+-
classification_head.dense.weight    | MISSING | 
classification_head.out_proj.weight | MISSING | 
classification_head.out_proj.bias   | MISSING | 
classification_head.dense.bias      | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



T5-large loaded in FP16.

READING DATASET

Total examples: 127400
Relevant (+1): 25562
Irrelevant (-1): 101838

DATASET INFORMATION
Examples: 127400
Batches: 127400
Batch size: 1
Gradient accumulation: 8
Effective batch size: 8
Maximum tokens: 384

PREPARING MODEL FOR GPU
GPU allocated: 4.51 GB
GPU reserved: 4.52 GB

Optimizer steps per epoch: 15925
Total optimizer steps: 127400
FP16 training: ENABLED

STARTING T5 FINE-TUNING

The model is learning:
Question + Retrieved Context
        ↓
     T5-large
        ↓
Relevant (+1) / Irrelevant (-1)




/tmp/ipykernel_826/577185332.py:704: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(


Epoch 1/8:   0%|          | 0/127400 [00:00<?, ?it/s]

/tmp/ipykernel_826/577185332.py:821: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(


ValueError: Attempting to unscale FP16 gradients.

In [ ]:
# ============================================================
# T5 EVALUATOR FINE-TUNING
# COMPLETE GOOGLE COLAB CODE
# ============================================================
#
# DATASET FORMAT:
#
# Question [SEP] Retrieved passage<TAB>1
# Question [SEP] Retrieved passage<TAB>0
#
# 1 = Relevant
# 0 = Irrelevant
#
# MODEL:
# T5-small
#
# TASK:
# Question + Passage -> 0 or 1
#
# CRAG SCORE:
# 1 -> +1
# 0 -> -1
#
# ============================================================


# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q -U transformers datasets sentencepiece accelerate scikit-learn


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import torch

from google.colab import files

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from datasets import Dataset

from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed
)


# ============================================================
# 3. CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

MODEL_NAME = "google-t5/t5-small"


# ------------------------------------------------------------
# MODEL SAVE DIRECTORY
# ------------------------------------------------------------

OUTPUT_DIR = "/content/t5_evaluator"


# ------------------------------------------------------------
# FINAL MODEL DIRECTORY
# ------------------------------------------------------------

FINAL_MODEL_DIR = "/content/t5_evaluator_final"


# ------------------------------------------------------------
# MAXIMUM INPUT LENGTH
#
# Your passages can be long.
#
# 384 reduces GPU memory usage.
#
# If this works successfully, we can later experiment with 512.
# ------------------------------------------------------------

MAX_INPUT_LENGTH = 384


# ------------------------------------------------------------
# OUTPUT LENGTH
#
# The model only needs to generate:
#
# 0
# or
# 1
# ------------------------------------------------------------

MAX_TARGET_LENGTH = 4


# ------------------------------------------------------------
# VALIDATION SPLIT
#
# 10% validation
# 90% training
# ------------------------------------------------------------

VALIDATION_SIZE = 0.10


# ------------------------------------------------------------
# RANDOM SEED
# ------------------------------------------------------------

SEED = 42

set_seed(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 4. CHECK GPU
# ============================================================

print("=" * 70)
print("GPU INFORMATION")
print("=" * 70)

if torch.cuda.is_available():

    device = torch.device("cuda")

    print("GPU available : YES")
    print("GPU name      :", torch.cuda.get_device_name(0))

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print(
        "GPU memory    :",
        round(gpu_memory, 2),
        "GB"
    )

else:

    device = torch.device("cpu")

    print("GPU available : NO")
    print("WARNING: Training on CPU will be very slow.")

print("=" * 70)


# ============================================================
# 5. UPLOAD DATASET
# ============================================================

print("\n")
print("=" * 70)
print("UPLOAD YOUR DATASET")
print("=" * 70)

print("\nA file selection window will appear.")
print("Select your dataset .txt file.")

uploaded = files.upload()

if len(uploaded) == 0:

    raise ValueError(
        "No dataset was uploaded."
    )


# ------------------------------------------------------------
# Get uploaded filename
# ------------------------------------------------------------

uploaded_filename = list(uploaded.keys())[0]

DATASET_PATH = "/content/" + uploaded_filename

print("\nDataset uploaded successfully.")

print("Filename:")
print(uploaded_filename)

print("\nDataset path:")
print(DATASET_PATH)


# ============================================================
# 6. VERIFY DATASET
# ============================================================

if not os.path.exists(DATASET_PATH):

    raise FileNotFoundError(
        "Dataset could not be found: "
        + DATASET_PATH
    )

print("\nDataset exists: YES")


# ============================================================
# 7. LOAD DATASET
# ============================================================

print("\n")
print("=" * 70)
print("LOADING DATASET")
print("=" * 70)

data = []

malformed_lines = 0


with open(
    DATASET_PATH,
    "r",
    encoding="utf-8"
) as file:

    for line_number, line in enumerate(
        file,
        start=1
    ):

        line = line.strip()

        # ----------------------------------------------------
        # Ignore empty lines
        # ----------------------------------------------------

        if not line:
            continue

        try:

            # ------------------------------------------------
            # IMPORTANT:
            #
            # Your dataset has:
            #
            # Question [SEP] Passage<TAB>Label
            #
            # We split from the RIGHT so that tabs inside
            # the passage do not break the parser.
            # ------------------------------------------------

            text, label = line.rsplit(
                "\t",
                1
            )

            label = label.strip()

            # ------------------------------------------------
            # Only accept binary labels
            # ------------------------------------------------

            if label not in ["0", "1"]:

                malformed_lines += 1

                continue

            # ------------------------------------------------
            # Separate question and passage
            # ------------------------------------------------

            if "[SEP]" not in text:

                malformed_lines += 1

                continue

            question, passage = text.split(
                "[SEP]",
                1
            )

            question = question.strip()
            passage = passage.strip()

            # ------------------------------------------------
            # Check empty values
            # ------------------------------------------------

            if not question or not passage:

                malformed_lines += 1

                continue

            # ------------------------------------------------
            # Add example
            # ------------------------------------------------

            data.append({

                "question": question,

                "passage": passage,

                "label": int(label)

            })

        except Exception:

            malformed_lines += 1


# ============================================================
# 8. CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(data)


print("\nDataset loading completed.")

print(
    "Valid examples:",
    len(df)
)

print(
    "Skipped/malformed lines:",
    malformed_lines
)


# ============================================================
# 9. CHECK DATASET
# ============================================================

if len(df) == 0:

    raise ValueError(
        "No valid examples were found in the dataset."
    )


print("\n")
print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("\nColumns:")
print(
    df.columns.tolist()
)


print("\nTotal examples:")
print(
    len(df)
)


print("\nLabel distribution:")
print(
    df["label"].value_counts()
)


print("\nLabel percentage:")
print(
    df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


# ============================================================
# 10. SHOW SAMPLE DATA
# ============================================================

print("\n")
print("=" * 70)
print("SAMPLE DATA")
print("=" * 70)


for i in range(
    min(3, len(df))
):

    print("\n")
    print(
        "Example",
        i + 1
    )

    print("-" * 70)

    print("\nQUESTION:")

    print(
        df.iloc[i]["question"]
    )

    print("\nPASSAGE:")

    print(
        df.iloc[i]["passage"][:500]
    )

    print("\nLABEL:")

    print(
        df.iloc[i]["label"]
    )


# ============================================================
# 11. TRAIN / VALIDATION SPLIT
# ============================================================

print("\n")
print("=" * 70)
print("TRAIN / VALIDATION SPLIT")
print("=" * 70)


train_df, val_df = train_test_split(

    df,

    test_size=VALIDATION_SIZE,

    random_state=SEED,

    # --------------------------------------------------------
    # Stratified split keeps the 0/1 distribution similar
    # in training and validation datasets.
    # --------------------------------------------------------

    stratify=df["label"]
)


train_df = train_df.reset_index(
    drop=True
)

val_df = val_df.reset_index(
    drop=True
)


print("\nTraining examples:")
print(
    len(train_df)
)

print("\nValidation examples:")
print(
    len(val_df)
)


print("\nTraining labels:")
print(
    train_df["label"].value_counts()
)


print("\nValidation labels:")
print(
    val_df["label"].value_counts()
)


# ============================================================
# 12. CONVERT TO HUGGING FACE DATASETS
# ============================================================

train_dataset = Dataset.from_pandas(

    train_df[
        [
            "question",
            "passage",
            "label"
        ]
    ],

    preserve_index=False
)


val_dataset = Dataset.from_pandas(

    val_df[
        [
            "question",
            "passage",
            "label"
        ]
    ],

    preserve_index=False
)


print("\nHugging Face datasets created successfully.")


# ============================================================
# 13. LOAD T5 TOKENIZER
# ============================================================

print("\n")
print("=" * 70)
print("LOADING T5 TOKENIZER")
print("=" * 70)


tokenizer = T5Tokenizer.from_pretrained(
    MODEL_NAME
)


print("\nTokenizer loaded successfully.")


# ============================================================
# 14. PREPROCESSING FUNCTION
# ============================================================

def preprocess_function(examples):

    inputs = []

    targets = []

    # --------------------------------------------------------
    # Process every example
    # --------------------------------------------------------

    for question, passage, label in zip(

        examples["question"],

        examples["passage"],

        examples["label"]

    ):

        # ----------------------------------------------------
        # T5 INPUT
        #
        # We give T5 an explicit instruction.
        #
        # Example:
        #
        # evaluate relevance:
        # question: What is George Rankin's occupation?
        # context: George Rankin was an Australian...
        # ----------------------------------------------------

        input_text = (

            "evaluate relevance: "

            "question: "
            + question

            + " context: "

            + passage

        )


        # ----------------------------------------------------
        # TARGET
        #
        # T5 must generate:
        #
        # 0
        #
        # or
        #
        # 1
        # ----------------------------------------------------

        target_text = str(label)


        inputs.append(
            input_text
        )

        targets.append(
            target_text
        )


    # ========================================================
    # TOKENIZE INPUT
    # ========================================================

    model_inputs = tokenizer(

        inputs,

        max_length=MAX_INPUT_LENGTH,

        truncation=True

    )


    # ========================================================
    # TOKENIZE TARGET
    # ========================================================

    labels = tokenizer(

        text_target=targets,

        max_length=MAX_TARGET_LENGTH,

        truncation=True

    )


    # --------------------------------------------------------
    # Add labels
    # --------------------------------------------------------

    model_inputs["labels"] = labels[
        "input_ids"
    ]


    return model_inputs


# ============================================================
# 15. TOKENIZE TRAINING DATA
# ============================================================

print("\n")
print("=" * 70)
print("TOKENIZING TRAINING DATA")
print("=" * 70)


tokenized_train = train_dataset.map(

    preprocess_function,

    batched=True,

    remove_columns=train_dataset.column_names,

    desc="Tokenizing training dataset"

)


print("\nTraining tokenization completed.")


# ============================================================
# 16. TOKENIZE VALIDATION DATA
# ============================================================

print("\n")
print("=" * 70)
print("TOKENIZING VALIDATION DATA")
print("=" * 70)


tokenized_val = val_dataset.map(

    preprocess_function,

    batched=True,

    remove_columns=val_dataset.column_names,

    desc="Tokenizing validation dataset"

)


print("\nValidation tokenization completed.")


# ============================================================
# 17. CLEAR UNUSED MEMORY
# ============================================================

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# 18. LOAD T5 MODEL
# ============================================================

print("\n")
print("=" * 70)
print("LOADING T5 MODEL")
print("=" * 70)


model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME
)


print("\nT5 model loaded successfully.")


print(
    "Number of parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)


# ============================================================
# 19. MEMORY OPTIMIZATION
# ============================================================

print("\n")
print("=" * 70)
print("ENABLING MEMORY OPTIMIZATION")
print("=" * 70)


# ------------------------------------------------------------
# Gradient checkpointing
#
# Reduces GPU memory usage.
# ------------------------------------------------------------

model.gradient_checkpointing_enable()


# ------------------------------------------------------------
# T5 cache must be disabled when gradient checkpointing
# is enabled.
# ------------------------------------------------------------

model.config.use_cache = False


# ------------------------------------------------------------
# Move model to GPU
# ------------------------------------------------------------

model.to(device)


print("\nGradient checkpointing: ENABLED")

print(
    "Model moved to:",
    device
)


# ============================================================
# 20. DATA COLLATOR
# ============================================================

#
# Dynamically pads examples within each batch.
#
# This saves GPU memory.
#

data_collator = DataCollatorForSeq2Seq(

    tokenizer=tokenizer,

    model=model,

    padding=True

)


print("\nData collator created.")


# ============================================================
# 21. HELPER FUNCTION FOR PREDICTIONS
# ============================================================

def convert_prediction_to_label(text):

    """
    Convert T5 generated text into a binary class.

    "1" -> 1
    "0" -> 0
    """

    text = text.strip()


    # --------------------------------------------------------
    # Exact prediction
    # --------------------------------------------------------

    if text == "1":

        return 1


    if text == "0":

        return 0


    # --------------------------------------------------------
    # Handle unexpected output
    #
    # Example:
    #
    # "The answer is 1"
    # --------------------------------------------------------

    if "1" in text:

        return 1


    if "0" in text:

        return 0


    # --------------------------------------------------------
    # Safe fallback
    # --------------------------------------------------------

    return 0


# ============================================================
# 22. METRICS FUNCTION
# ============================================================

def compute_metrics(eval_preds):

    predictions, labels = eval_preds


    # --------------------------------------------------------
    # Replace -100 values
    #
    # -100 represents ignored tokens.
    # --------------------------------------------------------

    labels = np.where(

        labels != -100,

        labels,

        tokenizer.pad_token_id

    )


    # --------------------------------------------------------
    # Convert generated token IDs to text
    # --------------------------------------------------------

    decoded_predictions = tokenizer.batch_decode(

        predictions,

        skip_special_tokens=True

    )


    # --------------------------------------------------------
    # Convert true token IDs to text
    # --------------------------------------------------------

    decoded_labels = tokenizer.batch_decode(

        labels,

        skip_special_tokens=True

    )


    # --------------------------------------------------------
    # Convert strings to 0/1
    # --------------------------------------------------------

    pred_classes = [

        convert_prediction_to_label(x)

        for x in decoded_predictions

    ]


    true_classes = [

        convert_prediction_to_label(x)

        for x in decoded_labels

    ]


    # ========================================================
    # CALCULATE METRICS
    # ========================================================

    accuracy = accuracy_score(

        true_classes,

        pred_classes

    )


    precision = precision_score(

        true_classes,

        pred_classes,

        zero_division=0

    )


    recall = recall_score(

        true_classes,

        pred_classes,

        zero_division=0

    )


    f1 = f1_score(

        true_classes,

        pred_classes,

        zero_division=0

    )


    return {

        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1

    }


# ============================================================
# 23. TRAINING CONFIGURATION
# ============================================================

print("\n")
print("=" * 70)
print("CREATING TRAINING CONFIGURATION")
print("=" * 70)


training_args = Seq2SeqTrainingArguments(

    # --------------------------------------------------------
    # Output folder
    # --------------------------------------------------------

    output_dir=OUTPUT_DIR,


    # --------------------------------------------------------
    # EPOCHS
    # --------------------------------------------------------

    num_train_epochs=3,


    # --------------------------------------------------------
    # GPU BATCH SIZE
    #
    # Small batch prevents CUDA OOM.
    # --------------------------------------------------------

    per_device_train_batch_size=2,

    per_device_eval_batch_size=2,


    # --------------------------------------------------------
    # GRADIENT ACCUMULATION
    #
    # Effective batch size:
    #
    # 2 x 8 = 16
    # --------------------------------------------------------

    gradient_accumulation_steps=8,


    # --------------------------------------------------------
    # LEARNING RATE
    # --------------------------------------------------------

    learning_rate=3e-4,


    # --------------------------------------------------------
    # WEIGHT DECAY
    # --------------------------------------------------------

    weight_decay=0.01,


    # --------------------------------------------------------
    # WARMUP
    #
    # IMPORTANT:
    #
    # We use warmup_steps instead of warmup_ratio
    # to avoid the warning from newer Transformers versions.
    # --------------------------------------------------------

    warmup_steps=500,


    # --------------------------------------------------------
    # EVALUATION
    # --------------------------------------------------------

    eval_strategy="steps",

    eval_steps=1000,


    # --------------------------------------------------------
    # SAVE CHECKPOINTS
    # --------------------------------------------------------

    save_strategy="steps",

    save_steps=1000,

    save_total_limit=2,


    # --------------------------------------------------------
    # LOGGING
    # --------------------------------------------------------

    logging_strategy="steps",

    logging_steps=100,


    # --------------------------------------------------------
    # T5 GENERATION DURING EVALUATION
    # --------------------------------------------------------

    predict_with_generate=True,

    generation_max_length=MAX_TARGET_LENGTH,


    # --------------------------------------------------------
    # FP16
    #
    # Reduces GPU memory usage on T4.
    # --------------------------------------------------------

    fp16=torch.cuda.is_available(),


    # --------------------------------------------------------
    # MEMORY-EFFICIENT T5 OPTIMIZER
    # --------------------------------------------------------

    optim="adafactor",


    # --------------------------------------------------------
    # LOAD BEST MODEL
    # --------------------------------------------------------

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,


    # --------------------------------------------------------
    # RANDOM SEED
    # --------------------------------------------------------

    seed=SEED,


    # --------------------------------------------------------
    # NO EXTERNAL LOGGING
    # --------------------------------------------------------

    report_to="none",


    # --------------------------------------------------------
    # SAFE FOR COLAB
    # --------------------------------------------------------

    dataloader_num_workers=0,


    # --------------------------------------------------------
    # REMOVE UNUSED COLUMNS
    # --------------------------------------------------------

    remove_unused_columns=True,


    # --------------------------------------------------------
    # GRADIENT CHECKPOINTING
    # --------------------------------------------------------

    gradient_checkpointing=True

)


print("\nTraining configuration created successfully.")


# ============================================================
# 24. CREATE SEQ2SEQ TRAINER
# ============================================================

print("\n")
print("=" * 70)
print("CREATING T5 TRAINER")
print("=" * 70)


trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_val,

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # New Transformers versions use:
    #
    # processing_class=tokenizer
    #
    # NOT:
    #
    # tokenizer=tokenizer
    #
    # This fixes the error you received.
    # --------------------------------------------------------

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics

)


print("\nTrainer created successfully.")


# ============================================================
# 25. START FINE-TUNING
# ============================================================

print("\n")
print("=" * 70)
print("STARTING T5 FINE-TUNING")
print("=" * 70)


print("\nTraining task:")

print(
    "Question + Retrieved Passage -> 0 / 1"
)


print("\nMeaning:")

print(
    "1 = Relevant"
)

print(
    "0 = Irrelevant"
)


print("\nTraining examples:")

print(
    len(train_df)
)


print("\nValidation examples:")

print(
    len(val_df)
)


print("\nEpochs:")

print(
    3
)


print("\nBatch size:")

print(
    2
)


print("\nGradient accumulation:")

print(
    8
)


print("\nEffective batch size:")

print(
    2 * 8
)


print("\nTraining is starting now...")

print("=" * 70)


# ============================================================
# THIS IS WHERE THE MODEL ACTUALLY STARTS TRAINING
# ============================================================

train_result = trainer.train()


# ============================================================
# 26. SAVE TRAINING RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)


print("\nTraining results:")

print(
    train_result
)


# ============================================================
# 27. FINAL EVALUATION
# ============================================================

print("\n")
print("=" * 70)
print("FINAL VALIDATION")
print("=" * 70)


evaluation_results = trainer.evaluate()


print("\nEvaluation results:")


for key, value in evaluation_results.items():

    print(
        f"{key}: {value}"
    )


# ============================================================
# 28. SAVE FINAL MODEL
# ============================================================

print("\n")
print("=" * 70)
print("SAVING FINAL MODEL")
print("=" * 70)


trainer.save_model(
    FINAL_MODEL_DIR
)


tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)


print("\nFinal model saved successfully.")

print(
    "Model location:"
)

print(
    FINAL_MODEL_DIR
)


# ============================================================
# 29. CLEAR MEMORY BEFORE RELOADING
# ============================================================

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# 30. LOAD FINE-TUNED MODEL
# ============================================================

print("\n")
print("=" * 70)
print("LOADING FINE-TUNED MODEL")
print("=" * 70)


evaluator_tokenizer = T5Tokenizer.from_pretrained(

    FINAL_MODEL_DIR

)


evaluator_model = T5ForConditionalGeneration.from_pretrained(

    FINAL_MODEL_DIR

)


evaluator_model.to(device)


evaluator_model.eval()


print("\nFine-tuned evaluator loaded successfully.")


# ============================================================
# 31. EVALUATOR FUNCTION
# ============================================================

def evaluate_retrieved_passage(

    question,

    passage

):

    """
    Evaluate whether a retrieved passage is relevant.

    Returns:

        prediction -> T5 generated text
        label      -> 0 or 1
        score      -> -1 or +1
    """


    # --------------------------------------------------------
    # Create T5 input
    # --------------------------------------------------------

    input_text = (

        "evaluate relevance: "

        "question: "
        + question

        + " context: "

        + passage

    )


    # --------------------------------------------------------
    # Tokenize
    # --------------------------------------------------------

    inputs = evaluator_tokenizer(

        input_text,

        return_tensors="pt",

        max_length=MAX_INPUT_LENGTH,

        truncation=True

    )


    # --------------------------------------------------------
    # Move tensors to GPU
    # --------------------------------------------------------

    inputs = {

        key: value.to(device)

        for key, value in inputs.items()

    }


    # --------------------------------------------------------
    # Generate prediction
    # --------------------------------------------------------

    with torch.no_grad():

        outputs = evaluator_model.generate(

            **inputs,

            max_length=MAX_TARGET_LENGTH,

            num_beams=1

        )


    # --------------------------------------------------------
    # Convert generated tokens to text
    # --------------------------------------------------------

    prediction_text = evaluator_tokenizer.decode(

        outputs[0],

        skip_special_tokens=True

    ).strip()


    # --------------------------------------------------------
    # Convert to 0/1
    # --------------------------------------------------------

    label = convert_prediction_to_label(

        prediction_text

    )


    # --------------------------------------------------------
    # Convert binary label to CRAG score
    #
    # Relevant   -> +1
    # Irrelevant -> -1
    # --------------------------------------------------------

    if label == 1:

        score = +1

    else:

        score = -1


    return {

        "prediction": prediction_text,

        "label": label,

        "score": score

    }


# ============================================================
# 32. TEST 1 - RELEVANT PASSAGE
# ============================================================

print("\n")
print("=" * 70)
print("TEST 1 - RELEVANT PASSAGE")
print("=" * 70)


test_question = (

    "What is George Rankin's occupation?"

)


test_passage = (

    "George Rankin was an Australian soldier and politician. "

    "He attended the local state school and became a farmer."

)


result = evaluate_retrieved_passage(

    test_question,

    test_passage

)


print("\nQuestion:")

print(
    test_question
)


print("\nPassage:")

print(
    test_passage
)


print("\nT5 generated output:")

print(
    result["prediction"]
)


print("\nPredicted label:")

print(
    result["label"]
)


print("\nCRAG evaluator score:")

print(
    result["score"]
)


# ============================================================
# 33. TEST 2 - IRRELEVANT PASSAGE
# ============================================================

print("\n")
print("=" * 70)
print("TEST 2 - IRRELEVANT PASSAGE")
print("=" * 70)


test_question_2 = (

    "What is George Rankin's occupation?"

)


test_passage_2 = (

    "Bangai-O Spirits is an action game for the Nintendo DS. "

    "The game has 160 levels and features a level editor."

)


result_2 = evaluate_retrieved_passage(

    test_question_2,

    test_passage_2

)


print("\nQuestion:")

print(
    test_question_2
)


print("\nPassage:")

print(
    test_passage_2
)


print("\nT5 generated output:")

print(
    result_2["prediction"]
)


print("\nPredicted label:")

print(
    result_2["label"]
)


print("\nCRAG evaluator score:")

print(
    result_2["score"]
)


# ============================================================
# 34. BATCH EVALUATION FUNCTION
# ============================================================

def evaluate_multiple_passages(

    question,

    passages

):

    """
    Evaluate multiple retrieved passages.

    Useful for top-k RAG results.
    """

    results = []


    for index, passage in enumerate(

        passages

    ):

        result = evaluate_retrieved_passage(

            question,

            passage

        )


        results.append({

            "passage_number":
                index + 1,

            "label":
                result["label"],

            "score":
                result["score"],

            "prediction":
                result["prediction"],

            "passage":
                passage

        })


    return results


# ============================================================
# 35. TEST TOP-K RETRIEVAL
# ============================================================

print("\n")
print("=" * 70)
print("TOP-K RAG EVALUATION TEST")
print("=" * 70)


question = (

    "What is George Rankin's occupation?"

)


retrieved_passages = [

    # --------------------------------------------------------
    # PASSAGE 1
    # Relevant
    # --------------------------------------------------------

    (
        "George Rankin was an Australian soldier and politician. "

        "He attended the local state school and became a farmer."
    ),


    # --------------------------------------------------------
    # PASSAGE 2
    # Irrelevant
    # --------------------------------------------------------

    (
        "Bangai-O Spirits is an action game for the Nintendo DS."
    ),


    # --------------------------------------------------------
    # PASSAGE 3
    # Different George Rankin
    # --------------------------------------------------------

    (
        "George Claus Rankin was a British judge in India."
    ),


    # --------------------------------------------------------
    # PASSAGE 4
    # Relevant
    # --------------------------------------------------------

    (
        "George Rankin was born at Bamawm, Victoria and "
        "became a farmer."
    )

]


results = evaluate_multiple_passages(

    question,

    retrieved_passages

)


for result in results:

    print("\n")
    print("-" * 70)

    print(
        "Passage:",
        result["passage_number"]
    )

    print(
        "T5 output:",
        result["prediction"]
    )

    print(
        "Label:",
        result["label"]
    )

    print(
        "CRAG score:",
        result["score"]
    )


# ============================================================
# 36. CONFUSION MATRIX
# ============================================================

print("\n")
print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)


prediction_output = trainer.predict(

    tokenized_val

)


predictions = prediction_output.predictions

labels = prediction_output.label_ids


# ------------------------------------------------------------
# Sometimes predictions can be returned as a tuple.
# ------------------------------------------------------------

if isinstance(
    predictions,
    tuple
):

    predictions = predictions[0]


# ------------------------------------------------------------
# Replace -100
# ------------------------------------------------------------

labels = np.where(

    labels != -100,

    labels,

    tokenizer.pad_token_id

)


# ------------------------------------------------------------
# Decode predictions
# ------------------------------------------------------------

decoded_predictions = tokenizer.batch_decode(

    predictions,

    skip_special_tokens=True

)


# ------------------------------------------------------------
# Decode labels
# ------------------------------------------------------------

decoded_labels = tokenizer.batch_decode(

    labels,

    skip_special_tokens=True

)


# ------------------------------------------------------------
# Convert predictions to classes
# ------------------------------------------------------------

pred_classes = [

    convert_prediction_to_label(x)

    for x in decoded_predictions

]


true_classes = [

    convert_prediction_to_label(x)

    for x in decoded_labels

]


# ------------------------------------------------------------
# Create confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(

    true_classes,

    pred_classes

)


print("\nConfusion Matrix:")

print(
    cm
)


# ============================================================
# 37. CLASSIFICATION REPORT
# ============================================================

print("\n")
print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)


print(

    classification_report(

        true_classes,

        pred_classes,

        target_names=[

            "Irrelevant (0)",

            "Relevant (1)"

        ],

        zero_division=0

    )

)


# ============================================================
# 38. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("T5 EVALUATOR TRAINING COMPLETED")
print("=" * 70)


print("\nBase model:")

print(
    MODEL_NAME
)


print("\nTraining examples:")

print(
    len(train_df)
)


print("\nValidation examples:")

print(
    len(val_df)
)


print("\nMaximum input length:")

print(
    MAX_INPUT_LENGTH
)


print("\nFinal model saved at:")

print(
    FINAL_MODEL_DIR
)


print("\n")
print("EVALUATOR LOGIC")
print("-" * 70)


print(
    "T5 output 1 -> Relevant -> CRAG score +1"
)


print(
    "T5 output 0 -> Irrelevant -> CRAG score -1"
)


print("\n")
print("=" * 70)
print("DONE")
print("=" * 70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 134.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 130.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.6 MB/s eta 0:00:00
GPU INFORMATION
GPU available : YES
GPU name      : Tesla T4
GPU memory    : 14.56 GB


UPLOAD YOUR DATASET

A file selection window will appear.
Select your dataset .txt file.


Saving train_popqa.txt to train_popqa (1).txt

Dataset uploaded successfully.
Filename:
train_popqa (1).txt

Dataset path:
/content/train_popqa (1).txt

Dataset exists: YES


LOADING DATASET

Dataset loading completed.
Valid examples: 127400
Skipped/malformed lines: 0


DATASET INFORMATION

Columns:
['question', 'passage', 'label']

Total examples:
127400

Label distribution:
label
0    101838
1     25562
Name: count, dtype: int64

Label percentage:
label
0    79.94
1    20.06
Name: proportion, dtype: float64


SAMPLE DATA


Example 1
----------------------------------------------------------------------

QUESTION:
What is George Rankin's occupation?

PASSAGE:
George Rankin Major General George James Rankin, (1 May 1887 – 28 December 1957) was an Australian soldier and politician. He served in both the House of Representatives and the Senate, representing the Country Party of Australia. Rankin was born at Bamawm, Victoria, the tenth child of Irish farmer James Rankin and Sarah, née Gal

Tokenizing training dataset:   0%|          | 0/114660 [00:00<?, ? examples/s]


Training tokenization completed.


TOKENIZING VALIDATION DATA


Tokenizing validation dataset:   0%|          | 0/12740 [00:00<?, ? examples/s]

...

Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1000,1.255702,0.128149,0.826060,0.553628,0.686620,0.612993
2000,1.062149,0.128248,0.831790,0.556313,0.798122,0.655632
3000,0.966825,0.117606,0.873783,0.748950,0.557903,0.639462
4000,0.937740,0.106641,0.882418,0.740455,0.637324,0.685029
5000,0.924051,0.103776,0.870487,0.670045,0.698357,0.683908


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]